# Berth planning by cost

How many berths should a terminal provide? More berths reduce vessel waiting but increase infrastructure and operating costs. This example enumerates candidate berth counts and minimises **annual berth cost + annual vessel waiting cost**, using both exact M/M/n theory and OpenQTSim estimates.

Traffic demand and service rate **per berth** remain fixed across candidates. All berths share one FIFO queue. Arrivals are Poisson and service times exponential; quay length, compatibility, downtime, priorities and reneging are not included. All cost inputs below are **illustrative teaching assumptions, not project estimates**.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import openqtsim
from IPython.display import display
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker, mmc_waiting_time_factor

print(f'This notebook uses OpenQTSim {openqtsim.__version__}')

This notebook uses OpenQTSim v0.5.1


## 1. Define traffic, capacity and annual costs

`annual_berth_cost` includes an **annualised** construction cost plus annual operating/maintenance cost per berth; do not insert a one-off construction amount directly. `waiting_cost_per_vessel_hour` is the economic cost of one vessel waiting for one hour.

`nr_arr` is the customers in one simulation and `nr_replications` is the independent runs per stable candidate. Their product is the number of simulated customers per candidate, **not** the annual traffic used in the cost model. Annual traffic is `arrival_rate * hours_per_year`.

The master seed is split into candidate seeds and then separate arrival/service seeds for every replication.

In [2]:
servers = list(range(1, 9))
arrival_rate = 2.4  # total vessels/hour, fixed for every candidate
mu_server = 1.0  # vessels/hour per berth, also fixed
hours_per_year = 365 * 24
annual_berth_cost = 1_500_000.0  # EUR/berth/year (annualised investment + operation)
waiting_cost_per_vessel_hour = 1_000.0  # EUR/vessel/hour waiting
nr_arr = 50_000
nr_replications = 4
seed = 2024
client = None  # optionally supply an existing Dask Client

if min(arrival_rate, mu_server, hours_per_year, annual_berth_cost, waiting_cost_per_vessel_hour) <= 0:
    raise ValueError('Traffic, capacity, annual hours and cost inputs must be positive.')
if nr_replications < 2:
    raise ValueError('Use at least two replications to estimate a standard error.')
annual_arrivals = arrival_rate * hours_per_year
print(f'Annual traffic: {annual_arrivals:,.0f} vessels')
print(f'{nr_arr * nr_replications:,} simulated customers per stable candidate')

Annual traffic: 21,024 vessels
200,000 simulated customers per stable candidate


## 2. Exclude unstable layouts and calculate the analytical costs

A candidate is stable only if $\rho_n = \lambda/(n\mu_{server}) < 1$. Unstable candidates have **no finite steady-state waiting cost** and must not look cheap simply because a finite simulation ends. They remain visible in the table with infinite waiting/total cost, but are not simulated or plotted.

For stable candidates, Erlang C gives $W_{q,n}$ in hours. Using Little's law, the long-run average number waiting is $L_{q,n}=\lambda W_{q,n}$. Thus

$$C_{wait,n}=\lambda H W_{q,n} c_{wait}, \qquad C_{total,n}=n c_{berth}+C_{wait,n},$$

where $H$ is annual operating hours. These are annualised **steady-state** costs, not the costs of the simulated start-up/draining period. Service-time costs are omitted because the mean service time and traffic are identical for every candidate.

In [3]:
planning = pd.DataFrame({'berths': servers}).set_index('berths')
planning['occupancy'] = arrival_rate / (planning.index * mu_server)
planning['stable'] = planning['occupancy'] < 1
planning['analytical W_q (hours)'] = [
    mmc_waiting_time_factor(int(n), row['occupancy']) / mu_server for n, row in planning.iterrows()
]
planning['berth cost (EUR/year)'] = planning.index * annual_berth_cost
planning['analytical waiting cost (EUR/year)'] = annual_arrivals * waiting_cost_per_vessel_hour * planning['analytical W_q (hours)']
planning['analytical total cost (EUR/year)'] = planning['berth cost (EUR/year)'] + planning['analytical waiting cost (EUR/year)']
stable_servers = planning.index[planning['stable']].tolist()
if not stable_servers:
    raise ValueError('No candidate is stable: add berths, increase service rate or reduce demand.')
display(planning.round(3))

,occupancy,stable,analytical W_q (hours),berth cost (EUR/year),analytical waiting cost (EUR/year),analytical total cost (EUR/year)
berths,,,,,,
1,2.400,False,inf,1500000.0,inf,inf
2,1.200,False,inf,3000000.0,inf,inf
3,0.800,True,1.079,4500000.0,2.267757e+07,2.717757e+07
4,0.600,True,0.179,6000000.0,3.771748e+06,9.771748e+06
5,0.480,True,0.044,7500000.0,9.178088e+05,8.417809e+06
6,0.400,True,0.011,9000000.0,2.333232e+05,9.233323e+06
7,0.343,True,0.003,10500000.0,5.736126e+04,1.055736e+07
8,0.300,True,0.001,12000000.0,1.328022e+04,1.201328e+07


## 3. Estimate waiting costs with simulation

For each candidate pass **total capacity** `n * mu_server` to `create_tasks`. Setting occupancy to `arrival_rate / (n * mu_server)` ensures that every simulation uses the same total arrival rate.

Each simulation starts empty and includes final draining. Its mean wait is a finite-run estimate of the stationary mean, not an exact steady-state sample. The reported standard error measures variability across independent replications; it does not cover start-up bias or uncertainty in demand, service rates and costs. More replications reduce sampling uncertainty; longer runs are also needed to assess start-up effects.

In [4]:
scenario_seeds = [int(child.generate_state(1)[0]) for child in np.random.SeedSequence(seed).spawn(len(stable_servers))]
tasks = []
for n, scenario_seed in zip(stable_servers, scenario_seeds):
    rho = arrival_rate / (n * mu_server)
    tasks.extend(create_tasks('M', 'M', [n], [rho], nr_arr, mu=n * mu_server,
                              nr_replications=nr_replications, seed=scenario_seed))
results = pd.DataFrame(run_tasks(tasks, client, func=statistics_worker))
estimates = results.groupby('c')['W_q'].agg(['mean', 'std', 'count'])
planning['simulated W_q (hours)'] = estimates['mean']
planning['W_q standard error (hours)'] = estimates['std'] / np.sqrt(estimates['count'])
planning['simulated waiting cost (EUR/year)'] = annual_arrivals * waiting_cost_per_vessel_hour * planning['simulated W_q (hours)']
planning['simulated total cost (EUR/year)'] = planning['berth cost (EUR/year)'] + planning['simulated waiting cost (EUR/year)']
planning['total cost standard error (EUR/year)'] = annual_arrivals * waiting_cost_per_vessel_hour * planning['W_q standard error (hours)']
display(planning.round(3))

,occupancy,stable,analytical W_q (hours),berth cost (EUR/year),analytical waiting cost (EUR/year),analytical total cost (EUR/year),simulated W_q (hours),W_q standard error (hours),simulated waiting cost (EUR/year),simulated total cost (EUR/year),total cost standard error (EUR/year)
berths,,,,,,,,,,,
1,2.400,False,inf,1500000.0,inf,inf,NaN,NaN,NaN,NaN,NaN
2,1.200,False,inf,3000000.0,inf,inf,NaN,NaN,NaN,NaN,NaN
3,0.800,True,1.079,4500000.0,2.267757e+07,2.717757e+07,1.086,0.019,2.283318e+07,2.733318e+07,390392.715
4,0.600,True,0.179,6000000.0,3.771748e+06,9.771748e+06,0.185,0.003,3.887020e+06,9.887020e+06,53546.916
5,0.480,True,0.044,7500000.0,9.178088e+05,8.417809e+06,0.043,0.001,9.044039e+05,8.404404e+06,16195.491
6,0.400,True,0.011,9000000.0,2.333232e+05,9.233323e+06,0.010,0.000,2.184517e+05,9.218452e+06,1063.830
7,0.343,True,0.003,10500000.0,5.736126e+04,1.055736e+07,0.003,0.000,5.639802e+04,1.055640e+07,2757.947
8,0.300,True,0.001,12000000.0,1.328022e+04,1.201328e+07,0.001,0.000,1.361282e+04,1.201361e+07,1482.778


## 4. Find the least-cost candidate

Enumerate the stable candidates rather than applying a continuous optimiser to an integer decision. Plot annual berth cost, waiting cost and their sum. The simulation markers show **one standard error** of the estimated total cost. An empirical minimum is not proof that one layout is better when differences are small relative to uncertainty.

The optimum is only over the candidate range and assumptions above. If a minimum occurs at the largest candidate, extend the range before claiming a global optimum.

In [5]:
stable = planning.loc[stable_servers]
analytical_optimum = int(stable['analytical total cost (EUR/year)'].idxmin())
simulated_optimum = int(stable['simulated total cost (EUR/year)'].idxmin())
print(f'Analytical least-cost candidate: {analytical_optimum} berths')
print(f'Simulation least-cost candidate: {simulated_optimum} berths')
print(f'Analytical annual cost at its minimum: EUR {stable.loc[analytical_optimum, "analytical total cost (EUR/year)"]:,.0f}')
if max(analytical_optimum, simulated_optimum) == max(servers):
    print('Minimum at upper candidate boundary: extend servers to check larger layouts.')

fig = go.Figure()
for column, label in [('berth cost (EUR/year)', 'Annual berth cost'),
                      ('analytical waiting cost (EUR/year)', 'Annual waiting cost (analytical)'),
                      ('analytical total cost (EUR/year)', 'Annual total cost (analytical)')]:
    fig.add_scatter(x=stable.index, y=stable[column] / 1e6, mode='lines+markers', name=label)
fig.add_scatter(x=stable.index, y=stable['simulated total cost (EUR/year)'] / 1e6, mode='markers',
                marker=dict(color='black', symbol='diamond', size=10), name='Annual total cost (simulation +/- 1 SE)',
                error_y=dict(type='data', array=stable['total cost standard error (EUR/year)'] / 1e6))
fig.add_vline(x=analytical_optimum, line_dash='dash', annotation_text=f'Analytical minimum: {analytical_optimum} berths')
fig.update_layout(title='Berth investment versus vessel waiting: illustrative annual costs',
                  xaxis=dict(title='Number of berths', dtick=1), yaxis_title='Annual cost (million EUR/year)',
                  template='plotly_white', height=600)
fig.show()

Analytical least-cost candidate: 5 berths
Simulation least-cost candidate: 5 berths
Analytical annual cost at its minimum: EUR 8,417,809


## 5. Planning interpretation

With the default teaching inputs the analytical optimum is **5 berths**. The minimum-capacity stable layout has only 3 berths, but its waiting cost outweighs the saving in infrastructure. Beyond the optimum, extra berths save too little waiting to justify their annual cost.

Try changing demand, service rate, annual berth cost and vessel waiting cost, then re-run all cells. More expensive waiting generally favours more berths; more expensive infrastructure favours fewer. Check robustness with longer simulations and more replications, especially when candidates have similar costs.

A real business case would also consider demand uncertainty and growth, discounting and phasing, heterogeneous vessels/berths, downtime, length constraints, and any service-level requirements. The optimum here is an illustration, not an investment recommendation.